# LangChain Trace 기록

- 구분: 정답 예시
- 원본: [`01_tracing_basic.py`](01_tracing_basic.py)
- 연결 교안: [4-2-langfuse-observability-and-evaluation.pdf](../../../../docs/raw/Chapter02/day02/4-2-langfuse-observability-and-evaluation.pdf)

원본 Python 코드는 수정 없이 아래의 한 코드 셀에 담았습니다. 설명과 실행 준비는 별도 셀입니다. **모두 실행** 대신 필요한 셀을 직접 선택하세요.


## 학습 목표

1. Trace와 CallbackHandler의 역할을 설명하고 콜백 연결을 확인한다.
2. 코드를 실행하기 전 입력, 출력, 외부 호출과 파일 변경 가능성을 설명한다.
3. CodeMate가 만든 결과를 원본과 교안에 비춰 검토한다.


## 코드를 읽는 순서

프롬프트 → LLM → 문자열 파서 → 단일 질문 → 여러 질문 → flush 순서로 읽는다.

교안은 개념과 목표를 설명합니다. 코드에서 실제 구현된 부분과 TODO를 구분하세요.


## 실행 준비

저장소 안에서 VS Code로 열고 프로젝트 `.venv` 커널을 선택합니다. 의존성은 `uv sync`로 준비합니다. 사내망에서는 사내 LLM 환경을, 사외에서는 허용된 provider를 준비합니다. 자격증명 값은 셀에 적지 않습니다. 아래 준비 셀은 저장소 루트를 찾고 `__file__`을 원본 Python 파일 경로로 설정합니다.


In [ ]:
from pathlib import Path
import os
import sys
_start = Path.cwd().resolve()
_repo = next((p for p in (_start, *_start.parents) if (p / "pyproject.toml").is_file() and (p / "config" / "llm.py").is_file()), None)
if _repo is None:
    raise FileNotFoundError("저장소 안에서 노트북을 열어 주세요.")
os.chdir(_repo)
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
__file__ = str(_repo / 'src/solution/Chapter02/05-observability/01_tracing_basic.py')
print(f"실행 기준 폴더: {_repo}")


## 원본 코드 — 직접 실행

**조건과 부작용:** LLM API와 Langfuse 서버에 질문·답변이 전송된다. `.env`의 `LANGFUSE_*` 설정을 확인한다. 원본과 정답본 모두 TODO가 남아 있다.

준비 셀을 먼저 실행하고 필요한 서비스·설정을 확인하세요. 다음 셀은 `manual-run`으로 표시했습니다. 원본 전체가 한 번에 실행되므로 API 요청, 파일 변경 또는 대기가 생길 수 있습니다.


In [ ]:
"""
LangChain 트레이싱: Langfuse로 LLM 호출 흐름 기록하기

핵심 개념:
  - 트레이싱 = LLM 호출의 입력·출력·레이턴시·토큰 수를 자동 기록
  - CallbackHandler를 체인에 붙이면 별도 코드 수정 없이 모든 단계가 추적됨
  - cloud.langfuse.com 대시보드에서 실시간 확인 가능

사전 준비:
  1. cloud.langfuse.com 에서 프로젝트 생성 후 API 키 발급
  2. .env 파일에 아래 세 줄 추가:
       LANGFUSE_PUBLIC_KEY=pk-lf-...
       LANGFUSE_SECRET_KEY=sk-lf-...
       LANGFUSE_BASE_URL=https://jp.cloud.langfuse.com

실행:
  python src/solution/Chapter02/05-observability/01_tracing_basic.py
"""

from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langfuse import get_client
from langfuse.langchain import CallbackHandler

from config.llm import get_llm

load_dotenv()

# CallbackHandler: .env의 LANGFUSE_* 환경변수를 자동으로 읽어 인증
# 이 핸들러를 chain.invoke()에 전달하면 모든 단계가 Langfuse에 자동 기록됨
langfuse_handler = CallbackHandler()

llm = get_llm()
prompt = ChatPromptTemplate.from_messages([
    ('system', '한국어로 간결하게 답변하세요.'),
    ('human', '{question}'),
])
chain = prompt | llm | StrOutputParser()

print('=== 단일 질문 트레이싱 ===')
# config={"callbacks": [...]} — LangChain이 실행 중 각 단계마다 핸들러를 호출함
# TODO: chain.invoke를 호출하면서, config의 callback으로 `langfuse_handler`를 전달하세요.
result = chain.invoke(
    {'question': 'LangGraph와 LangChain의 차이를 한 줄로 설명해줘'},
    config={'callbacks': [langfuse_handler]},
)
print(result)
print()

print('=== 여러 질문 배치 트레이싱 ===')
questions = [
    {'question': 'RAG가 뭔가요?'},
    {'question': 'Vector DB는 왜 쓰나요?'},
]
# 각 invoke마다 별도의 trace로 기록됨
for q in questions:
    # TODO: chain.invoke를 호출하면서, config의 callback으로 `langfuse_handler`를 전달하세요.
    answer = chain.invoke(q, config={'callbacks': [langfuse_handler]})
    print(f"Q: {q['question']}")
    print(f"A: {answer}")
    print()

# flush: 비동기로 큐에 쌓인 이벤트를 Langfuse 서버로 전송 완료 후 종료
# 스크립트 종료 직전에 호출하지 않으면 일부 데이터가 누락될 수 있음
langfuse = get_client()
langfuse.flush()
print('Tracing에서 확인')


## CodeMate에게 요청하고 검토하기

> 두 `chain.invoke` 위치에서 Langfuse 콜백이 실제로 전달되는지 검토하고 Trace 누락 원인을 알려 줘.

CodeMate가 제안한 수정의 근거를 읽고, 실행 가능한 환경에서 출력·Trace·생성 파일을 확인하세요.


## 확인 질문

1. 콜백 객체를 만들기만 하면 기록되는가? flush를 생략하면 무엇이 누락될 수 있는가?
2. 실행 전 어떤 환경과 자료를 확인해야 하는가?
3. 결과가 기대와 다르면 어느 단계의 기록부터 볼 것인가?
